# Fake image dataset

Builds the **synthetic half** of the prototype sample, at `data/fakes/`, to pair with the authentic
half that [authentics.ipynb](authentics.ipynb) writes to `data/authentic/`.

Target: **2 images from each of the 36 benchmark generators = 72 fakes**, against 72 authentic
images. That keeps both of the benchmark's balance properties at once — exact 50/50 class balance,
and uniform per-generator coverage, which is what per-generator score models need.

### Why this reads parquet, and why it re-encodes

The fake part is on the Hub as `lrzpellegrini/AI-GenBench-fake_part`, but two things about it need
handling.

**It is stored in native formats, not JPEG.** AI-GenBench's assembly script ships with
`make_jpeg_dataset = False` (`dataset_creation/simple_dataset_generation.py`), so neither half is
normalized. In the 72-image draw below that comes out as roughly 38 JPEG, 33 PNG and 1 WEBP. Writing
those bytes verbatim next to an all-JPEG authentic half would make the two classes separable on
container format alone — about 46% of the fakes would be losslessly coded and none of the reals
would be. That is the "Fake or JPEG?" confound (Grommelt et al.) in its crudest form.

AI-GenBench's other setting, `make_jpeg_dataset = True`, pushes every image through
`prepare_image(convert_to_jpeg=True, jpeg_quality=95)`. That is exactly what
[authentics.ipynb](authentics.ipynb) already does, so this notebook applies the **identical function**
to the fakes. Both halves then carry a single quality-95 encode derived from their own original, and
compression history stops being a class cue.

**Getting the original bytes is the expensive part.** Two obvious routes are both wrong:

- **`load_dataset(..., streaming=True).shuffle(...)`** pulls whole shards. The validation split is
  **7.0 GB across 15 shards** (~486 MB each), and shuffling opens several at once.
- **`datasets-server`'s `/rows` endpoint**, which hands back ready-to-fetch image URLs, is far
  cheaper — but it **re-encodes everything at JPEG quality 75**. Images from four different origin
  datasets all come back with the identical libjpeg-default quantization table
  `[8, 6, 5, 8, 12, 20, 26, 31]`. Normalizing from those would leave a q75 generation baked into the
  fakes that the reals never went through.

So we go to the parquet for the true stored bytes. Parquet is columnar and `image.bytes` is ~100% of
the file, so reading the metadata columns costs almost nothing; only the row groups we actually draw
from get downloaded, at ~17 MB per 100 images.

## Setup

In [ ]:
import io
import json
import os
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))

AIGENBENCH = PROJECT_ROOT / "AI-GenBench"
OUT_DIR = PROJECT_ROOT / "data" / "fakes"
IMG_DIR = OUT_DIR / "images"
AUTHENTIC_DIR = PROJECT_ROOT / "data" / "authentic"

REPO = "lrzpellegrini/AI-GenBench-fake_part"
SPLIT = "validation"
SHARD = "validation-00000-of-00015.parquet"  # any one shard holds all 36 generators

N_PER_GENERATOR = 2   # 2 x 36 generators = 72, matching the authentic half
MAX_ROW_GROUPS = 8    # guard: each row group is ~17 MB, so cap the worst case at ~136 MB
FAKE_LABEL = 1

IMG_DIR.mkdir(parents=True, exist_ok=True)
print("writing to", OUT_DIR)

In [ ]:
if not AIGENBENCH.exists():
    !git clone --depth 1 https://github.com/MI-BioLab/AI-GenBench.git {AIGENBENCH}

# Resolve against PROJECT_ROOT instead of %cd-ing into the repo: cd-ing from inside AI-GenBench/ is
# what produced the nested AI-GenBench/AI-GenBench/ clone.
sys.path.insert(0, str(AIGENBENCH / "training_and_evaluation"))
from ai_gen_bench_metadata.benchmark_generators import BENCHMARK_GENERATORS

# The generator registry: release date per generator. This is the spine of the thesis's G<=T
# construction, so it is worth keeping in view rather than buried in the repo.
print(len(BENCHMARK_GENERATORS), "generators,", 
      f"{min(BENCHMARK_GENERATORS.values())} to {max(BENCHMARK_GENERATORS.values())}")
for name, date in BENCHMARK_GENERATORS.items():
    print(f"  {date}  {name}")

## Index the shard

Reading only the metadata columns is effectively free — `image.bytes` is the whole payload, and every
other column rounds to zero. This gives us a row-to-generator map before committing to any download.

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem

filesystem = HfFileSystem()
parquet = pq.ParquetFile(filesystem.open(f"datasets/{REPO}/data/{SHARD}", "rb"))
metadata = parquet.metadata

ROWS_PER_GROUP = metadata.row_group(0).num_rows
MB_PER_GROUP = metadata.row_group(0).total_byte_size / 1e6
print(f"{metadata.num_rows} rows in {metadata.num_row_groups} row groups "
      f"of {ROWS_PER_GROUP}, ~{MB_PER_GROUP:.0f} MB each")

index = parquet.read(
    columns=["generator", "file_id", "width", "height", "origin_dataset", "description"]
).to_pandas()
index["row_group"] = index.index // ROWS_PER_GROUP

print(f"indexed {len(index)} rows, {index['generator'].nunique()} distinct generators")

## Choose the row groups

Rows are shuffled with respect to generator rather than grouped by it, so a single 100-row group
already covers most of the 36. Take groups in order until every generator has enough rows to draw
from, and print the bill before paying it.

In [ ]:
selected, available = [], Counter()
for group in range(metadata.num_row_groups):
    if all(available[name] >= N_PER_GENERATOR for name in BENCHMARK_GENERATORS):
        break
    if len(selected) == MAX_ROW_GROUPS:
        break
    selected.append(group)
    available.update(index.loc[index["row_group"] == group, "generator"])

short = {name: available[name] for name in BENCHMARK_GENERATORS
         if available[name] < N_PER_GENERATOR}
if short:
    raise RuntimeError(
        f"still short of {N_PER_GENERATOR} after {len(selected)} row groups "
        f"(cap {MAX_ROW_GROUPS}): {short}"
    )

print(f"row groups {selected} cover all {len(BENCHMARK_GENERATORS)} generators "
      f"-> ~{len(selected) * MB_PER_GROUP:.0f} MB to download")

## Download and normalize

`read_row_groups` pulls only the chosen groups. Each stored image is then decoded and re-encoded
through the same `prepare_image` port the authentic notebook uses, so both halves land on one
quality-95 JPEG regardless of what the source container was.

In [ ]:
from PIL import Image

JPEG_QUALITY = 95  # AI-GenBench dataset_utils/common_utils.py


def prepare_image(image):
    """Port of AI-GenBench prepare_image(convert_to_jpeg=True). Returns encoded JPEG bytes.

    Byte-for-byte the same function authentics.ipynb applies to the real half — that sameness is the
    point, so edit both or neither.
    """
    image.info.pop("xmp", None)
    if image.mode != "RGB":
        if image.mode != "RGBA":
            image = image.convert("RGBA")
        background = Image.new("RGBA", image.size, (255, 255, 255))
        image = Image.alpha_composite(background, image).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=JPEG_QUALITY)
    return buffer.getvalue()

In [ ]:
# Deterministic pick: the first N_PER_GENERATOR rows of each generator, no seed needed.
picked = (
    index[index["row_group"].isin(selected) & index["generator"].isin(BENCHMARK_GENERATORS)]
    .groupby("generator", sort=False)
    .head(N_PER_GENERATOR)
)
print(f"{len(picked)} rows picked across {picked['generator'].nunique()} generators")

table = parquet.read_row_groups(selected, columns=["image", "file_id"])
stored = {
    file_id: record["bytes"]
    for file_id, record in zip(table.column("file_id").to_pylist(),
                               table.column("image").to_pylist())
}

rows, source_formats = [], Counter()
for record in picked.itertuples():
    with Image.open(io.BytesIO(stored[record.file_id])) as image:
        source_format = image.format
        jpeg = prepare_image(image)
    source_formats[source_format] += 1

    path = IMG_DIR / (record.file_id.replace("/", "_") + ".jpg")
    path.write_bytes(jpeg)
    rows.append({
        "file_id": record.file_id,
        "origin_dataset": record.origin_dataset,
        "label": FAKE_LABEL,
        "generator": record.generator,
        "description": record.description,
        "width": record.width,
        "height": record.height,
        "source_format": source_format,
        "path": str(path.relative_to(PROJECT_ROOT)),
    })

written_mb = sum(p.stat().st_size for p in IMG_DIR.glob("*.jpg")) / 1e6
print(f"wrote {len(rows)} fake images, {written_mb:.1f} MB on disk")
print("source containers before normalizing:",
      ", ".join(f"{fmt} {n}" for fmt, n in source_formats.most_common()))

## Compression parity

The check that justifies the expensive path. Both halves should now carry the *same* quantization
table, because both were produced by the same `prepare_image` at quality 95. If they diverge, some
image skipped normalization — a class cue that would not show up anywhere in the manifest — so this
fails loudly rather than warning.

In [ ]:
Q75_LUMA = (8, 6, 5, 8, 12, 20, 26, 31)  # libjpeg default, i.e. a datasets-server re-encode


def luma_table(path):
    with Image.open(path) as image:
        assert image.format == "JPEG", f"{path.name} is {image.format}, not JPEG"
        return tuple(image.quantization[0][:8])


fake_tables = {luma_table(p) for p in sorted(IMG_DIR.glob("*.jpg"))}
assert len(fake_tables) == 1, f"fakes are not uniformly encoded: {fake_tables}"
assert Q75_LUMA not in fake_tables, "fakes carry the q75 default table — a datasets-server re-encode"
print(f"fakes      qtab0[:8] = {list(next(iter(fake_tables)))}  ({len(list(IMG_DIR.glob('*.jpg')))} images)")

authentic_images = sorted((AUTHENTIC_DIR / "images").glob("*.jpg"))
if not authentic_images:
    print("no authentic images yet — run authentics.ipynb to compare the two halves")
else:
    authentic_tables = {luma_table(p) for p in authentic_images}
    print(f"authentics qtab0[:8] = {list(next(iter(authentic_tables)))}  ({len(authentic_images)} images)")
    assert authentic_tables == fake_tables, (
        f"compression history differs between halves: {authentic_tables} vs {fake_tables}"
    )
    print("both halves share one quantization table — compression history is not a class cue")

## Manifest

In [ ]:
import pandas as pd

manifest = pd.DataFrame(rows)
manifest["release_date"] = manifest["generator"].map(BENCHMARK_GENERATORS)
manifest = manifest.sort_values(["release_date", "file_id"]).reset_index(drop=True)

manifest.to_parquet(OUT_DIR / f"fakes_{SPLIT}.parquet", index=False)
with open(OUT_DIR / f"fakes_{SPLIT}.jsonl", "w") as handle:
    for row in manifest.to_dict("records"):
        handle.write(json.dumps(row) + "\n")

per_generator = manifest["generator"].value_counts()
print(f"{len(manifest)} fakes over {len(per_generator)} generators, "
      f"{per_generator.min()}-{per_generator.max()} each")
print(manifest["source_format"].value_counts().to_string())
manifest[["release_date", "generator", "origin_dataset", "source_format", "width", "height"]].head(10)

In [ ]:
import matplotlib.pyplot as plt

sample = manifest.drop_duplicates("generator").head(6)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(PROJECT_ROOT / row["path"]))
    ax.set_title(f'{row["generator"]}  ({row["release_date"]})\n{row["width"]}x{row["height"]}',
                 fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

## Balance check

Both halves together, as the benchmark would have it.

In [ ]:
authentic = pd.read_parquet(AUTHENTIC_DIR / f"authentic_{SPLIT}.parquet")
combined = pd.concat([authentic, manifest], ignore_index=True)

counts = combined["label"].value_counts().sort_index()
print(f"authentic (label 0): {counts.get(0, 0)}")
print(f"fake      (label 1): {counts.get(1, 0)}")
assert counts.get(0, 0) == counts.get(1, 0), "halves are not balanced — re-run authentics.ipynb"

print()
print(combined.groupby(["label", "origin_dataset"]).size().to_string())

## Notes

- **The sample is balanced, but not paired.** AI-GenBench gives priority to *aligned* reals — the
  `paired_real_images` column, where a real image shares a caption, segmentation mask, or inpainting
  source with a specific fake — so the two classes match on content as well as on count. Both
  notebooks sample independently, so that alignment is absent here. Any real-vs-fake score
  comparison on this sample is still confounded by content, and restoring the pairing is the next
  thing to do before trusting a separation number.
- **Compression history is matched**, which is the one confound this notebook does control: both
  halves are a single quality-95 JPEG encode from AI-GenBench's own `prepare_image`. This is the
  `make_jpeg_dataset = True` variant of the benchmark. The shipped default is `False`, which leaves
  the fakes as native PNG/JPEG/WEBP against reals that are almost all JPEG — reproducible from here
  by writing `stored[...]` verbatim, but not something to score against without accounting for it.
- **`source_format` is kept in the manifest** so the pre-normalization container is still visible.
  Worth checking that detector scores do not track it; if they do, normalization did not fully close
  the gap.
- **One shard, validation split.** All 36 generators appear in every shard (55-87 rows each), so a
  single shard is enough at this sample size. Scaling up means widening to more shards, not more row
  groups from this one.
- **`release_date` comes from `BENCHMARK_GENERATORS`**, the registry the thesis's G<=T aggregation is
  built on. Note the benchmark stops at 2024-08 (FLUX 1), which is the coverage gap newer generators
  would have to fill.